<a href="https://colab.research.google.com/github/dynamodenis/QDrant/blob/main/Documentation%20engine%20project/QdrantDocumentationEngine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Production-Ready Documentation Search Engine

You’ll build a sophisticated documentation search engine that shows hybrid retrieval, multivector reranking, and production-quality evaluation.

Your search engine will understand both semantic meaning and exact keywords, then use fine-grained reranking to surface the most relevant documentation sections. When someone searches for “how to configure HNSW parameters,” your system should return the exact section with practical examples, not just a page that mentions “HNSW” somewhere.

This mirrors real-world retrieval challenges where users need precise answers from large documentation sets. You’ll implement the complete pipeline: ingestion with smart chunking, hybrid search with dense and sparse signals, and multivector reranking for precision.

## Setup & Dependencies

Install required packages for working with Qdrant, embeddings, and numpy.

In [ ]:
# Install dependencies
!pip install -q "qdrant-client~=1.15.1" "fastembed~=0.7.3" numpy requests beautifulsoup4 markdownify lxml llama_index llama-index-embeddings-huggingface

# CLient and project set up

In [4]:
from qdrant_client import QdrantClient, models
from google.colab import userdata
import numpy as np

client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))

/tmp/ipykernel_1187/3084920485.py:5: UserWarning: Qdrant client version 1.15.1 is incompatible with server version 1.17.1. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))


# Collection Design

Create docs_search with three vectors (dense 384, sparse, ColBERT multivector with m=0). If you choose a 768-dim dense model, set size=768 below.

In [8]:
collection_name = "docs_search"


# Delete collection if it exists (for clean restart)
try:
    client.delete_collection(collection_name)
    print(f"Deleted existing collection: {collection_name}")
except Exception:
    pass  # Collection doesn't exist, which is fine

client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": models.VectorParams(
            distance=models.Distance.COSINE,
            size=384,
            quantization_config=models.ScalarQuantization(
                # Use scalar since we are using low dimensions and binary qauntitization requires size > 1000 dims
                scalar=models.ScalarQuantizationConfig(
                    type=models.ScalarType.INT8,
                    quantile=0.99,  # ignore top 1% outliers for better compression
                    always_ram=True  # keep quantized vectors in RAM for speed
                )
            ),
            # Tune HNSW for better recall vs speed tradeoff
            hnsw_config=models.HnswConfigDiff(
                m=0, # disable during ingestion,
                ef_construct=100,
            )
        ),
        "colbert": models.VectorParams(
            distance=models.Distance.COSINE,
            size=128,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM
            ),
            hnsw_config=models.HnswConfigDiff(m=0), # Reranking only
            # Store on disk since we will use it for reranking and we need speed
            on_disk=False
        )
    },
    sparse_vectors_config={"sparse": models.SparseVectorParams(
        index=models.SparseIndexParams(
            on_disk=False, # Keep index in Ram though expensive
        ),
        modifier=models.Modifier.IDF
    )},
     # Also tell optimizer not to index during upload
    optimizers_config=models.OptimizersConfigDiff(
        indexing_threshold=0,  # disable indexing during upload entirely
    ),
)

Deleted existing collection: docs_search


True

# Parse and Chunk Documents

Pick a documentation set and parse it into structured sections. Preserve the hierarchy users expect.

Section-based chunking

1. Primary unit: one chunk per section
2. Context: store adjacent sections in prev_section_text / next_section_text
3. Metadata: keep titles, URLs, and breadcrumbs for attribution and navigation

In [ ]:
import requests
import time
import re
from bs4 import BeautifulSoup
from markdownify import markdownify as md


def get_qdrant_urls():
    sitemap_url = "https://qdrant.tech/sitemap.xml"
    response = requests.get(sitemap_url)
    soup = BeautifulSoup(response.content, 'xml')
    urls = [loc.text for loc in soup.find_all('loc')]
    return [url for url in urls if "/documentation/" in url]


def extract_breadcrumbs(soup: BeautifulSoup) -> list[str]:
    """
    Extract breadcrumb trail from the page.
    Tries multiple common breadcrumb patterns.
    """
    # Try common breadcrumb selectors
    for selector in [
        "nav[aria-label='breadcrumb']",
        ".breadcrumb",
        ".breadcrumbs",
        "[class*='breadcrumb']",
    ]:
        breadcrumb = soup.select_one(selector)
        if breadcrumb:
            items = breadcrumb.find_all(["a", "li", "span"])
            crumbs = [item.get_text(strip=True) for item in items if item.get_text(strip=True)]
            if crumbs:
                return crumbs

    # Fallback: derive from URL structure
    return []


def extract_tags(soup: BeautifulSoup, url: str, section: str) -> list[str]:
    """
    Extract tags from multiple sources:
    1. Meta keywords tag
    2. URL path segments
    3. Section name from URL
    4. Article category classes
    """
    tags = set()

    # 1. Meta keywords
    meta_keywords = soup.find("meta", {"name": "keywords"})
    if meta_keywords and meta_keywords.get("content"):
        for kw in meta_keywords["content"].split(","):
            tags.add(kw.strip().lower())

    # 2. Meta tags (og:tags or article tags)
    for meta in soup.find_all("meta", {"property": re.compile("article:")}):
        if meta.get("content"):
            tags.add(meta["content"].strip().lower())

    # 3. URL path segments — these are meaningful for Qdrant docs
    path_parts = url.replace("https://qdrant.tech/documentation/", "").split("/")
    for part in path_parts:
        if part and part not in ["", "index"]:
            # Convert hyphens to spaces, clean up
            tags.add(part.replace("-", " ").strip())

    # 4. Section from URL
    if section:
        tags.add(section.replace("-", " "))

    # Remove very generic tags
    generic = {"documentation", "qdrant", "tech", ""}
    tags = tags - generic

    return sorted(list(tags))


def extract_section_anchors(soup: BeautifulSoup) -> dict[str, str]:
    """
    Build a mapping of heading text -> anchor URL fragment.
    e.g. "HNSW Parameters" -> "#hnsw-parameters"
    Used to build section_url for each chunk.
    """
    anchors = {}
    for heading in soup.find_all(["h1", "h2", "h3", "h4"]):
        # Headings often have an id or a child anchor tag
        heading_id = heading.get("id")
        if not heading_id:
            anchor_tag = heading.find("a", {"class": re.compile("anchor|header")})
            if anchor_tag:
                href = anchor_tag.get("href", "")
                heading_id = href.lstrip("#")

        if heading_id:
            anchors[heading.get_text(strip=True)] = f"#{heading_id}"

    return anchors


def scrape_page(url: str) -> dict:
    """
    Fetch a doc page and extract:
    - Clean Markdown text
    - Page title, section
    - Breadcrumbs
    - Tags
    - Section anchors (for building section_url per chunk)
    """
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except Exception as e:
        print(f"Failed to fetch {url}: {e}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # ── Extract metadata BEFORE removing elements ──────────
    breadcrumbs  = extract_breadcrumbs(soup)
    section_anchors = extract_section_anchors(soup)

    # ── Remove noise elements ───────────────────────────────
    for tag in soup.find_all(["nav", "footer", "header", "script", "style"]):
        tag.decompose()
    for tag in soup.find_all(class_=re.compile("sidebar|toc|navigation|breadcrumb")):
        tag.decompose()

    # ── Get main content ────────────────────────────────────
    main = soup.find("article") or soup.find("main") or soup.find("body")
    if not main:
        return None

    # ── Convert to Markdown ─────────────────────────────────
    markdown_text = md(
        str(main),
        heading_style="ATX",
        code_language_callback=lambda el: (
            el.get('class', [''])[0].replace('language-', '')
            if el.has_attr('class') else ''
        ),
        strip=['a', 'img']
    )

    # Clean excessive newlines
    clean_markdown = "\n".join([
        line.rstrip()
        for line in markdown_text.splitlines()
        if line.strip() or line == ""
    ])

    # ── Extract page title ──────────────────────────────────
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else url.split("/")[-2]

    # ── Extract section from URL ────────────────────────────
    parts = url.replace("https://qdrant.tech/documentation/", "").split("/")
    section = parts[0] if parts else "general"

    # ── Extract tags ────────────────────────────────────────
    tags = extract_tags(soup, url, section)

    # ── Build breadcrumbs fallback from URL ─────────────────
    if not breadcrumbs:
        breadcrumbs = [
            p.replace("-", " ").title()
            for p in parts if p
        ]

    return {
        "url":              url,
        "title":            title,
        "section":          section,
        "text":             clean_markdown,
        "breadcrumbs":      breadcrumbs,
        "tags":             tags,
        "section_anchors":  section_anchors,  # passed to chunker
    }


# ── Scrape all pages ────────────────────────────────────────
raw_docs = []
all_urls = get_qdrant_urls()

for i, url in enumerate(all_urls):
    print(f"[{i+1}/{len(all_urls)}] {url.split('documentation/')[-1]}", end=" ")
    doc = scrape_page(url)

    if doc is None:
        print(f"Did not find any document for {url}")
        continue

    raw_docs.append(doc)
    print(f"{len(doc['text'])} chars, {len(doc['tags'])} tags")
    time.sleep(0.5)

print(f"\n Scraped {len(raw_docs)} pages")

# Semantic Chunking

This is a documentation search so we need full meaning when doing the search and
Since I'm already using Markdown, the headings (#, ##, ###) are natural semantic boundaries — they are literally the document's own semantic structure. A ## Create a Collection section is a complete semantic unit.

In [ ]:
# Layer Chunking Strategy
import re
from llama_index.core import Document
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

DENSE_MODEL_ID = "BAAI/bge-small-en-v1.5"   # speed, 384-dim, 512 token limit
# DENSE_MODEL_ID = "BAAI/bge-base-en-v1.5"  # quality 768-dim, 512 token limit
SPARSE_MODEL_ID  = "Qdrant/bm25"              # bm25 sparse
COLBERT_MODEL_ID = "colbert-ir/colbertv2.0"                  # 128-dim multivector

# Load the semantic splitter once — reuse across all docs
# Uses the SAME model as your dense retriever for consistency
semantic_splitter = SemanticSplitterNodeParser(
    buffer_size=1,                        # compare adjacent sentences
    breakpoint_percentile_threshold=95,   # only cut at big topic shifts
    embed_model=HuggingFaceEmbedding(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
)

SEMANTIC_TRIGGER_WORDS = 200   # if section > 200 words, check semantics
MAX_WORDS = 380                # hard limit for embedding model


def layer1_structural_split(text: str) -> list[dict]:
    """
    Layer 1: Split by Markdown headings.
    Returns list of {heading_context, content} dicts.
    Each section is a candidate chunk.
    """
    heading_pattern = re.compile(r'^(#{1,6}\s+.+)$', re.MULTILINE)
    parts = re.split(heading_pattern, text)

    sections = []
    current_heading = "Introduction"
    heading_stack = []

    def get_level(h):
        m = re.match(r'^(#+)', h)
        return len(m.group(1)) if m else 0

    for part in parts:
        part = part.strip()
        if not part:
            continue
        if re.match(r'^#{1,6}\s+', part):
            level = get_level(part)
            heading_stack = [h for h in heading_stack if get_level(h) < level]
            heading_stack.append(part)
            current_heading = "\n".join(heading_stack)
        else:
            if part:
                sections.append({
                    "heading_context": current_heading,
                    "content": part,
                    "word_count": len(part.split())
                })

    return sections


def layer2_sentence_split(content: str, heading_context: str) -> list[str]:
    """
    Layer 2: Split long sections by sentences with overlap.
    Used when section exceeds MAX_WORDS but is semantically uniform
    (no major topic shifts detected by layer 3).
    """
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', content)
    chunks = []
    current_sentences = []
    current_words = len(heading_context.split())

    for sentence in sentences:
        s_words = len(sentence.split())
        if current_words + s_words > MAX_WORDS and current_sentences:
            chunk = heading_context + "\n\n" + " ".join(current_sentences)
            chunks.append(chunk.strip())
            current_sentences = current_sentences[-2:]  # 2-sentence overlap
            current_words = len(heading_context.split()) + sum(
                len(s.split()) for s in current_sentences
            )
        current_sentences.append(sentence)
        current_words += s_words

    if current_sentences:
        chunk = heading_context + "\n\n" + " ".join(current_sentences)
        chunks.append(chunk.strip())

    return chunks


def layer3_semantic_split(content: str, heading_context: str) -> list[str]:
    """
    Layer 3: Semantic chunking via embedding similarity.
    Used ONLY when a section is:
      - Long enough to potentially contain multiple topics
      - Structural/sentence splitting would cut mid-concept

    This is the most expensive layer — runs a neural model
    to find natural semantic breaks.
    """
    try:
        nodes = semantic_splitter.get_nodes_from_documents(
            [Document(text=content)]
        )
        chunks = []
        for node in nodes:
            node_text = node.text.strip()
            if not node_text:
                continue
            # Always prepend heading context so chunk is self-contained
            chunk = heading_context + "\n\n" + node_text
            # If semantic chunk is still too long → fall back to sentence split
            if len(chunk.split()) > MAX_WORDS:
                chunks.extend(layer2_sentence_split(node_text, heading_context))
            else:
                chunks.append(chunk.strip())
        return chunks
    except Exception as e:
        # If semantic split fails for any reason → fall back gracefully
        print(f"Semantic split failed, falling back to sentence split: {e}")
        return layer2_sentence_split(content, heading_context)


def chunk_document(text: str) -> list[str]:
    """
    Main chunking function, applies all 3 layers intelligently.

    Decision logic:

     Layer 1: Always runs structural Markdown split
              each heading section = candidate chunk

     Section fits in MAX_WORDS?
      YES  use as-is (Layer 1 result is enough)
      NO   is it > SEMANTIC_TRIGGER_WORDS?
            YES  try Layer 3 semantic split
            NO   use Layer 2 sentence split

    """
    # Layer 1 — always runs
    sections = layer1_structural_split(text)
    all_chunks = []

    for section in sections:
        content = section["content"]
        heading = section["heading_context"]
        word_count = section["word_count"]

        # ── Case 1: Section fits → use directly ─────────────
        if word_count <= MAX_WORDS:
            chunk = heading + "\n\n" + content
            all_chunks.append(chunk.strip())

        # ── Case 2: Long section → semantic split ───────────
        elif word_count > SEMANTIC_TRIGGER_WORDS:
            print(f"  Semantic split on section ({word_count} words): "
                  f"{heading.split(chr(10))[-1][:50]}...")
            semantic_chunks = layer3_semantic_split(content, heading)
            all_chunks.extend(semantic_chunks)

        # ── Case 3: Medium section → sentence split ─────────
        else:
            sentence_chunks = layer2_sentence_split(content, heading)
            all_chunks.extend(sentence_chunks)

    # Merge orphaned tiny chunks
    return merge_short_chunks(all_chunks, min_words=40)


def merge_short_chunks(chunks: list[str], min_words: int = 40) -> list[str]:
    """Merge chunks too short to carry semantic meaning."""
    merged = []
    buffer = ""
    for chunk in chunks:
        if buffer and len(buffer.split()) < min_words:
            buffer = buffer + "\n\n" + chunk
        else:
            if buffer:
                merged.append(buffer.strip())
            buffer = chunk
    if buffer:
        merged.append(buffer.strip())
    return merged

In [ ]:
# Run chunking on all docs
def chunk_document_with_payload(doc: dict) -> list[dict]:
    """
    Chunks the document AND builds the full rich payload per chunk.
    Adds: section_title, section_url, prev_section_text, next_section_text.
    """
    text             = doc["text"]
    page_url         = doc["url"]
    page_title       = doc["title"]
    section          = doc["section"]
    breadcrumbs      = doc["breadcrumbs"]
    tags             = doc["tags"]
    section_anchors  = doc["section_anchors"]

    # ── Run 3-layer chunking ─────────────────────────────────
    raw_chunks = chunk_document(text)   # your existing function

    if not raw_chunks:
        return []

    # ── Build payload for each chunk ────────────────────────
    result = []
    total  = len(raw_chunks)

    for j, chunk_text in enumerate(raw_chunks):

        # ── Extract section title from chunk's first heading ─
        # Every chunk starts with heading context e.g:
        # "# Collections\n## Create a collection\n\n..."
        heading_lines = [
            line for line in chunk_text.split("\n")
            if line.startswith("#")
        ]
        # Most specific heading = last one in the stack
        section_title = (
            heading_lines[-1].lstrip("#").strip()
            if heading_lines else page_title
        )
        # Top-level heading = first one
        top_heading = (
            heading_lines[0].lstrip("#").strip()
            if heading_lines else page_title
        )

        # ── Build section_url with anchor ───────────────────
        anchor = section_anchors.get(section_title, "")
        section_url = page_url + anchor

        # ── Build breadcrumbs for this specific chunk ────────
        # Start from page breadcrumbs and add section title
        chunk_breadcrumbs = breadcrumbs.copy()
        if section_title and section_title != page_title:
            if not chunk_breadcrumbs or chunk_breadcrumbs[-1] != section_title:
                chunk_breadcrumbs.append(section_title)

        # ── Prev / Next section text for context ────────────
        # These give the LLM surrounding context at query time
        prev_section_text = (
            # strip heading lines, keep just the content
            "\n".join([
                l for l in raw_chunks[j - 1].split("\n")
                if not l.startswith("#")
            ]).strip()[:500]        # cap at 500 chars
            if j > 0 else ""
        )
        next_section_text = (
            "\n".join([
                l for l in raw_chunks[j + 1].split("\n")
                if not l.startswith("#")
            ]).strip()[:500]
            if j < total - 1 else ""
        )

        # ── Extract just the body text (no heading lines) ───
        body_lines = [l for l in chunk_text.split("\n") if not l.startswith("#")]
        body_text  = "\n".join(body_lines).strip()

        # ── Augment tags with heading keywords ──────────────
        chunk_tags = tags.copy()
        for h in heading_lines:
            words = h.lstrip("#").strip().lower().split()
            chunk_tags.extend([w for w in words if len(w) > 3])
        chunk_tags = sorted(list(set(chunk_tags)))

        # ── Full payload ─────────────────────────────────────
        payload = {
            "page_title":        page_title,
            "section_title":     section_title,
            "top_heading":       top_heading,
            "page_url":          page_url,
            "section_url":       section_url,
            "breadcrumbs":       chunk_breadcrumbs,
            "chunk_text":        body_text,          # clean body, no headings
            "full_chunk":        chunk_text,         # full chunk with headings (for embedding)
            "prev_section_text": prev_section_text,
            "next_section_text": next_section_text,
            "tags":              chunk_tags,
            "section":           section,
            "chunk_index":       j,
            "total_chunks":      total,
        }
        result.append(payload)

    return result


# ── Run enriched chunking on all docs ───────────────────────
all_chunks = []

for doc in raw_docs:
    print(f"\nChunking: {doc['title']}")
    payloads = chunk_document_with_payload(doc)
    all_chunks.extend(payloads)

print(f"\n{'='*50}")
print(f"Total pages:    {len(raw_docs)}")
print(f"Total chunks:   {len(all_chunks)}")
print(f"Avg per page:   {len(all_chunks) / len(raw_docs):.1f}")

# ── Preview one payload ──────────────────────────────────────
import json
print("\n--- Sample Payload ---")
print(json.dumps(all_chunks[0], indent=2))

# Embed and Ingest

Embed and upload points with all three vectors and the payload fields you need for display and filters.

In [ ]:

# Load embedding models
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding

print("Loading embedding models (first run downloads from HuggingFace)...")
dense_model   = TextEmbedding(DENSE_MODEL_ID)
sparse_model  = SparseTextEmbedding(SPARSE_MODEL_ID)
colbert_model = LateInteractionTextEmbedding(COLBERT_MODEL_ID)

# Detect dense dimensions automatically
# so collection config always matches the model
sample = list(dense_model.embed(["test"]))[0]
DENSE_DIM = len(sample)

print("All models loaded")

In [ ]:
import numpy as np

# We are going to process points using batch and bulk uploads
BATCH_SIZE = 64  # How many chunks to embed & upload at once
all_points = []
total_chunks = len(all_chunks)

#Processing in Batches
for i in range(0, total_chunks, BATCH_SIZE):
    batch = all_chunks[i : i + BATCH_SIZE]

    # Extract just the text to embed from this batch
    batch_texts = [chunk["full_chunk"] for chunk in batch]

    print(f"Processing batch {i//BATCH_SIZE + 1}... ({i}/{total_chunks})")

    # FastEmbed .embed() is most efficient when passed a LIST of strings
    # We convert the generators to lists immediately
    dense_batch   = list(dense_model.embed(batch_texts))
    sparse_batch  = list(sparse_model.embed(batch_texts))
    colbert_batch = list(colbert_model.embed(batch_texts))

    # Build PointStructs for this batch
    for j, chunk in enumerate(batch):
        point = models.PointStruct(
            id=i + j, # Use the global index as ID
            vector={
                "dense":   dense_batch[j].tolist(),
                "sparse":  sparse_batch[j].as_object(),
                "colbert": colbert_batch[j].tolist(),
            },
            payload={
                    "page_title":        chunk["page_title"],
                    "section_title":     chunk["section_title"],
                    "page_url":          chunk["page_url"],
                    "section_url":       chunk["section_url"],
                    "breadcrumbs":       chunk["breadcrumbs"],
                    "chunk_text":        chunk["chunk_text"], #Real content for ui
                    "prev_section_text": chunk["prev_section_text"],
                    "next_section_text": chunk["next_section_text"],
                    "tags":              chunk["tags"],
                    "section":           chunk["section"],
                    "chunk_index":       chunk["chunk_index"],
                    "total_chunks":      chunk["total_chunks"],
                }
        )
        all_points.append(point)


# Batch upload of points

In [ ]:
"""
Tried to use client.upload_point() for bulk inserts but
I got an error of storing too much data per batch because of chunk_text
which is too large. SO we upload a batch of 16 per batch which works
"""
BATCH_SIZE = 16

print(f"Uploading {len(all_points)} points to Qdrant...")

for i in range(0, len(all_points), BATCH_SIZE):
    batch = all_points[i : i + BATCH_SIZE]

    print(f"{min(i + BATCH_SIZE, len(all_points))}/{len(all_points)}")
    try:
        client.upsert(
            collection_name=collection_name,
            points=batch,
            wait=True
        )
        print(f"{min(i + BATCH_SIZE, len(all_points))}/{len(all_points)}")

    except Exception as e:
        print(f"Batch {i//BATCH_SIZE + 1} failed: {e}")

# Re-enable HNSW & build index

After all uploads is done, apply index on a token thrshold of 1000 since our dataset only produces 2400 vectors.

In [41]:
# Now rebuild the dense HNSW graph on the complete dataset
client.update_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": models.VectorParamsDiff(
            hnsw_config=models.HnswConfigDiff(
                m=16,
                ef_construct=100,
            )
        )
    },
    optimizers_config=models.OptimizersConfigDiff(
        indexing_threshold=1000,
    ),
)

print("HNSW index building started...")

# Wait for indexing to complete
import time
while True:
    info = client.get_collection(collection_name)
    status = info.status
    if status.value == "green":
        print(f"Collection ready!")
        break
    print(f"Status: {status.value} indexing in progress...")
    time.sleep(3)

HNSW index building started...
Status: yellow indexing in progress...
Collection ready!


# Search Pipeline

Convert user queries into the three vector representations needed for hybrid search. Use Qdrant’s Universal Query API to combine dense and sparse with server-side fusion.

In [44]:
print("Testing search...")
test_query = "how to configure HNSW for better recall"

dense_q   = list(dense_model.embed([test_query]))[0]
sparse_q  = list(sparse_model.embed([test_query]))[0].as_object()
colbert_q = list(colbert_model.embed([test_query]))[0]

results = client.query_points(
    collection_name=collection_name,
    prefetch=[
        models.Prefetch(query=dense_q,  using="dense",  limit=20),
        models.Prefetch(query=sparse_q, using="sparse", limit=20),
    ],
    query=colbert_q,   # ColBERT reranks the prefetch results
    using="colbert",
    limit=3,
    with_payload=True,
)


Testing search...


# Format display with a nice UI

In [48]:
import textwrap

def format_snippet(text: str, query: str, max_length: int = 200) -> str:
    """
    Extract the most relevant part of the chunk text around query keywords.
    Highlights where the match likely is rather than just truncating from start.
    """
    if not text:
        return "No preview available."

    # Find the best window containing query keywords
    query_words = [w.lower() for w in query.split() if len(w) > 3]
    text_lower  = text.lower()

    best_pos = 0
    best_hits = 0

    # Slide a window across the text to find densest keyword region
    window = max_length
    for i in range(0, len(text) - window, 20):
        window_text = text_lower[i : i + window]
        hits = sum(1 for word in query_words if word in window_text)
        if hits > best_hits:
            best_hits = hits
            best_pos  = i

    # Extract snippet from best position
    snippet = text[best_pos : best_pos + max_length].strip()

    # Clean up — don't start mid-word
    if best_pos > 0 and not text[best_pos - 1].isspace():
        first_space = snippet.find(" ")
        snippet = snippet[first_space + 1:] if first_space != -1 else snippet

    # Add ellipsis if we're not at the boundaries
    prefix = "..." if best_pos > 0 else ""
    suffix = "..." if (best_pos + max_length) < len(text) else ""

    return f"{prefix}{snippet}{suffix}"

def score_to_label(score: float) -> str:
    """Convert numeric score to human-readable confidence label"""
    if score >= 0.85:   return "🟢 Excellent match"
    elif score >= 0.70: return "🟡 Good match"
    elif score >= 0.50: return "🟠 Partial match"
    else:               return "🔴 Weak match"


def display_results(results, query: str):
    """Transform raw Qdrant results into user-friendly output"""

    points = results.points

    print("=" * 65)
    print(f"  🔍 Query: \"{query}\"")
    print(f"  📄 {len(points)} results found")
    print("=" * 65)

    if not points:
        print("  No results found. Try a different query.")
        return

    for rank, point in enumerate(points, start=1):
        p = point.payload

        # ── Extract fields safely ──────────────────────────────────
        title       = p.get("page_title", "Untitled")
        section_raw = p.get("section_title", "unknown")
        url         = p.get("page_url", "#")
        chunk_text  = p.get("chunk_text", "")
        chunk_index = p.get("chunk_index", 0)
        total       = p.get("total_chunks", "?")
        score       = point.score

        # ── Format each field ──────────────────────────────────────
        snippet       = format_snippet(chunk_text, query, max_length=220)
        match_label   = score_to_label(score)

        # ── Display ────────────────────────────────────────────────
        print(f"\n  #{rank}  {title}")
        print(f"  {'─' * 61}")
        print(f"  📂 Section  : {section_raw}")
        print(f"  🔗 URL      : {url}")
        print(f"  📊 Score    : {score:.4f}  {match_label}")
        print(f"  📑 Chunk    : {chunk_index + 1} of {total}")
        print(f"\n  💬 Snippet  :")

        # Wrap snippet text neatly at 60 chars
        wrapped = textwrap.fill(snippet, width=60, initial_indent="     ",
                                subsequent_indent="     ")
        print(wrapped)
        print()

    print("=" * 65)
    print(f"  Developed by Dynamo Denis Mbugua https://www.linkedin.com/in/dynamo-denis-mbugua-53304b197/")
    print(f"  Powered by: Dense + Sparse (SPLADE) → ColBERT reranking")
    print("=" * 65)


# ── RUN THE SEARCH ─────────────────────────────────────────────────────────────
print("Testing search...\n")
test_query = "how to configure HNSW for better recall"

dense_q   = list(dense_model.embed([test_query]))[0]
sparse_q  = list(sparse_model.embed([test_query]))[0].as_object()
colbert_q = list(colbert_model.embed([test_query]))[0]

results = client.query_points(
    collection_name=collection_name,
    prefetch=[
        models.Prefetch(query=dense_q,  using="dense",  limit=20),
        models.Prefetch(query=sparse_q, using="sparse", limit=20),
    ],
    query=colbert_q,
    using="colbert",
    limit=5,
    with_payload=True,
)

display_results(results, test_query)

Testing search...

  🔍 Query: "how to configure HNSW for better recall"
  📄 5 results found

  #1  Indexing
  ─────────────────────────────────────────────────────────────
  📂 Section  : Vector Index
  🔗 URL      : https://qdrant.tech/documentation/manage-data/indexing/
  📊 Score    : 8.4127  🟢 Excellent match
  📑 Chunk    : 20 of 26

  💬 Snippet  :
     ...of HNSW index     # traversal for better
     performance.     # Note: 1Kb = 1 vector of size 256
     full_scan_threshold: 10000 ```  And so in the process
     of creating a collection. The `ef` parameter is
     configured during...


  #2  Optimizing Qdrant Performance: Three Scenarios
  ─────────────────────────────────────────────────────────────
  📂 Section  : Improving Precision
  🔗 URL      : https://qdrant.tech/documentation/operations/optimize/
  📊 Score    : 8.3263  🟢 Excellent match
  📑 Chunk    : 5 of 12

  💬 Snippet  :
     Increase the `ef` and `m` parameters of the HNSW index
     to improve precision, even with lim

# Analyse and Evaluate

Create evaluation data compare with real results

In [56]:
ground_truth_examples = [
    {
        "query": "how to configure HNSW parameters for better recall",
        "expected_urls": [
            "https://qdrant.tech/documentation/manage-data/indexing/",
            "https://qdrant.tech/documentation/operations/optimize/"
        ],
        "query_type": "how-to",
        "intent": "Technical configuration for accuracy/speed tradeoff."
    },
    {
        "query": "scalar vs binary quantization memory reduction",
        "expected_urls": [
            "https://qdrant.tech/documentation/manage-data/quantization/"
        ],
        "query_type": "concept",
        "intent": "Comparison of compression methods."
    },
    {
        "query": "create collection with replication factor and shards",
        "expected_urls": [
            "https://qdrant.tech/documentation/operations/distributed_deployment/"
        ],
        "query_type": "api-usage",
        "intent": "Distributed setup and reliability."
    },
    {
        "query": "combining vector search with boolean payload filters",
        "expected_urls": [
            "https://qdrant.tech/documentation/search/filtering/"
        ],
        "query_type": "troubleshooting",
        "intent": "Complex filtering logic on metadata."
    },
    {
        "query": "restore a collection from a remote snapshot url",
        "expected_urls": [
            "https://qdrant.tech/documentation/operations/snapshots/"
        ],
        "query_type": "how-to",
        "intent": "Disaster recovery and data migration."
    }
]


def evaluate_retrieval(client, collection_name, ground_truth):
    total_mrr = 0
    eval_results = []

    print(f"{'Rank':<6} | {'Score':<8} | {'Status':<12} | {'Query'}")
    print("-" * 80)

    for example in ground_truth:
        query = example["query"]
        expected_urls = [url.lower() for url in example["expected_urls"]]

        # 1. Generate Embeddings (Same as your search logic)
        dense_q   = list(dense_model.embed([query]))[0]
        sparse_q  = list(sparse_model.embed([query]))[0].as_object()
        colbert_q = list(colbert_model.embed([query]))[0]

        # 2. Execute Hybrid Search
        results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                models.Prefetch(query=dense_q,  using="dense",  limit=50),
                models.Prefetch(query=sparse_q, using="sparse", limit=50),
            ],
            query=colbert_q,
            using="colbert",
            limit=5, # Look at top 10 for evaluation
            with_payload=True,
        )

        # 3. Calculate Reciprocal Rank
        rr = 0
        found_rank = None
        for rank, point in enumerate(results.points, start=1):
            actual_url = point.payload.get("page_url", "").lower()

            # Check if actual URL matches any of our expected URLs
            if any(expected in actual_url for expected in expected_urls):
                rr = 1.0 / rank
                found_rank = rank
                break

        total_mrr += rr
        status = f"Pos {found_rank}" if found_rank else "❌ Not Found"
        print(f"{str(found_rank or '>10'):<6} | {rr:<8.3f} | {status:<12} | {query}")

    final_mrr = total_mrr / len(ground_truth)
    print("-" * 80)
    print(f"FINAL MEAN RECIPROCAL RANK: {final_mrr:.4f}")
    return final_mrr

# Run it
mrr_score = evaluate_retrieval(client, collection_name, ground_truth_examples)

Rank   | Score    | Status       | Query
--------------------------------------------------------------------------------
1      | 1.000    | Pos 1        | how to configure HNSW parameters for better recall
2      | 0.500    | Pos 2        | scalar vs binary quantization memory reduction
1      | 1.000    | Pos 1        | create collection with replication factor and shards
2      | 0.500    | Pos 2        | combining vector search with boolean payload filters
1      | 1.000    | Pos 1        | restore a collection from a remote snapshot url
--------------------------------------------------------------------------------
FINAL MEAN RECIPROCAL RANK: 0.8000
